# Methods 5 – Transformer Encoder

## Objective
In this assignment, I'm building a neural network that can identify positive and negative movie reviews.
I'll use the IMDB dataset and add parts of a Transformer encoder. In the end, I'll test the model to see how well it works.

In [1]:

import keras
from keras.datasets import imdb

# Dataset settings
max_features = 10000
max_len = 250

# Load IMDB movie reviews
(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=max_features
)

# Make all reviews the same length
x_train = keras.utils.pad_sequences(
    x_train, maxlen=max_len, padding="post"
)

x_test = keras.utils.pad_sequences(
    x_test, maxlen=max_len, padding="post"
)

print("Training data:", x_train.shape)
print("Test data:", x_test.shape)
print("First five labels:", y_train[:5])


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training data: (25000, 250)
Test data: (25000, 250)
First five labels: [1 0 0 1 0]


## Token and positional embedding
In this step, I turn token IDs into token vectors. I also create a vector for each position so the model can take word order into account. Finally, I add the token and position vectors together.

In [2]:

from keras.layers import Layer, Embedding

class TokenAndPositionEmbedding(Layer):
    def __init__(self, seq_len, vocab_size, emb_dim):
        super().__init__()

        self.token_emb = Embedding(
            input_dim=vocab_size,
            output_dim=emb_dim
        )

        self.pos_emb = Embedding(
            input_dim=seq_len,
            output_dim=emb_dim
        )

    def call(self, x_input):
        seq_len = keras.ops.shape(x_input)[-1]

        positions = keras.ops.arange(
            start=0, stop=seq_len, step=1
        )

        positions = self.pos_emb(positions)
        x_input = self.token_emb(x_input)

        return x_input + positions


# Test the embedding layer
embedding_layer = TokenAndPositionEmbedding(
    max_len, max_features, 32
)

sample = embedding_layer(x_train[:2])
print("Embedding shape:", sample.shape)


Embedding shape: (2, 250, 32)
